In [1]:
import os
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
imdb-dataset-of-top-1000-movies-and-tv-shows.zip: Skipping, found more recently modified local copy (use --force to force download)


In [2]:
import zipfile

zip_file = "imdb-dataset-of-top-1000-movies-and-tv-shows.zip"
extraction_path = "imdb_data"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Extracted files: {os.listdir(extraction_path)}")

Extracted files: ['imdb-dataset-of-top-1000-movies-and-tv-shows.zip', 'imdb_top_1000.csv']


In [3]:
import pandas as pd

df = pd.read_csv('imdb_data/imdb_top_1000.csv')

# Implement the Subsampling Toggle
USE_SUBSAMPLE = False
if USE_SUBSAMPLE:
    df = df.sample(frac=0.3, random_state=6)

# Select the star columns
stars_df = df[['Star1', 'Star2', 'Star3', 'Star4']]

# Convert to a list of lists (baskets), dropping NaN values
baskets = []
for index, row in stars_df.iterrows():
    # Drop missing values and convert to a list
    basket = [star for star in row if pd.notna(star)]
    if basket: # Only add if the basket isn't empty
        baskets.append(basket)

In [4]:
def apriori(baskets, support_threshold):
    
    baskets = list(baskets) # otherwise it gets exhausted after the first pass

    # ==========================================
    # FIRST PASS
    # ==========================================
    names2int = {}
    id_counter = 1
    item_counts = {}

    for basket in baskets:
        for star in basket:
            if star not in names2int:
                names2int[star] = id_counter
                id_counter += 1
                item_counts[names2int[star]] = 1
            else:
                item_counts[names2int[star]] += 1

    # ==========================================
    # BETWEEN TWO PASSES
    # ==========================================
    freq_items = {}
    new_numbering = 1

    for item in item_counts:
        if item_counts[item] >= support_threshold:
            freq_items[item] = new_numbering
            new_numbering += 1
        else:
            freq_items[item] = 0

    # ==========================================
    # SECOND PASS
    # ==========================================
    pairs_freq_items = []
    for basket in baskets:
        temp_freq = []
        for item in basket:
            if freq_items[names2int[item]] != 0:
                temp_freq.append(names2int[item])

        n = len(temp_freq)
        for i in range(n):
            for j in range(i + 1, n):
                # with min/max each pair is sorted in ascending order to avoid that (A, B) =/ (B, A)
                item1 = min(temp_freq[i], temp_freq[j])
                item2 = max(temp_freq[i], temp_freq[j])

                pairs_freq_items.append((item1, item2))

        temp_freq.clear()

    counts_of_pairs = {}

    for pair in pairs_freq_items:
        if pair not in counts_of_pairs:
            counts_of_pairs[pair] = 0
        counts_of_pairs[pair] += 1

    frequent_pairs = []

    for pair, count in counts_of_pairs.items():
        if count >= support_threshold:
            actor1 = next(name for name, integer in names2int.items() if integer == pair[0]) # to go back to names
            actor2 = next(name for name, integer in names2int.items() if integer == pair[1])

            frequent_pairs.append((actor1, actor2))

    return frequent_pairs

In [5]:
apriori_stars_pairs = apriori(baskets, support_threshold=3)

In [6]:
def pcy(baskets, support_threshold, num_buckets=10000):

    baskets = list(baskets)

    # ==========================================
    # FIRST PASS
    # ==========================================
    names2int = {}
    id_counter = 1
    item_counts = {}
    buckets = [0] * num_buckets

    for basket in baskets:
        for star in basket:
            if star not in names2int:
                names2int[star] = id_counter
                id_counter += 1
                item_counts[names2int[star]] = 1
            else:
                item_counts[names2int[star]] += 1

        basket_ids = [names2int[star] for star in basket]
        n_items = len(basket_ids)
        for i in range(n_items):
            for j in range(i + 1, n_items):
                item1 = min(basket_ids[i], basket_ids[j])
                item2 = max(basket_ids[i], basket_ids[j])

                bucket_idx = hash((item1, item2)) % num_buckets

                buckets[bucket_idx] += 1

    # ==========================================
    # BETWEEN TWO PASSES
    # ==========================================
    freq_items = {}
    new_numbering = 1

    for item in item_counts:
        if item_counts[item] >= support_threshold:
            freq_items[item] = new_numbering
            new_numbering += 1
        else:
            freq_items[item] = 0

    bitmap = bytearray(num_buckets)

    for i in range(num_buckets):
        if buckets[i] >= support_threshold:
            bitmap[i] = 1
        else:
            bitmap[i] = 0

    del buckets #substituted by the bitmap

    # ==========================================
    # SECOND PASS
    # ==========================================
    pairs_freq_items = []
    for basket in baskets:
        temp_freq = []
        for item in basket:
            if freq_items[names2int[item]] != 0:
                temp_freq.append(names2int[item])

        n = len(temp_freq)
        for i in range(n):
            for j in range(i + 1, n):
                item1 = min(temp_freq[i], temp_freq[j])
                item2 = max(temp_freq[i], temp_freq[j])

                bucket_idx = hash((item1, item2)) % num_buckets

                if bitmap[bucket_idx] == 1:
                    pairs_freq_items.append((item1, item2))

        temp_freq.clear()

    counts_of_pairs = {}

    for pair in pairs_freq_items:
        if pair not in counts_of_pairs:
            counts_of_pairs[pair] = 0
        counts_of_pairs[pair] += 1

    frequent_pairs = []

    for pair, count in counts_of_pairs.items():
        if count >= support_threshold:
            actor1 = next(name for name, integer in names2int.items() if integer == pair[0])
            actor2 = next(name for name, integer in names2int.items() if integer == pair[1])
            frequent_pairs.append((actor1, actor2))

    return frequent_pairs

In [7]:
pcy_stars_pairs = pcy(baskets, support_threshold=3, num_buckets=len(baskets)-1)

In [8]:
def SON(baskets, support_threshold, num_chunks=4):

    baskets = list(baskets)
    num_baskets = len(baskets)

    # ===========================
    # Create chunks
    # ===========================

    chunk_size = num_baskets // num_chunks
    chunks = []

    for basket in baskets:
        temp_basket = []
        for i in range(chunk_size):
            temp_basket.append(basket[i])
        chunks.append(temp_basket)

    p = 1.0 / len(chunks)
    local_threshold = support_threshold * p

    # ==========================
    # FIRST PASS
    # ==========================

